In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
import matplotlib.pyplot as plt
import numpy as np

In [3]:
dir=r"E:\dataset"

In [4]:
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dir,
    validation_split=0.2,
    subset="both",
    seed=123,
    image_size=(128,128),
    batch_size=32
)

train_ds, val_ds = dataset

Found 4092 files belonging to 2 classes.
Using 3274 files for training.
Using 818 files for validation.


In [11]:
train_ds.class_names


['with_mask', 'without_mask']

In [6]:
val_ds.class_names

['with_mask', 'without_mask']

In [5]:
model=Sequential([
    tf.keras.layers.Rescaling(1./255,input_shape=(128,128,3)),

    Conv2D(28,(3,3),activation='relu'),
    MaxPooling2D(),

    Conv2D(32,(3,3),activation='relu'),
    MaxPooling2D(),

    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D(),

    Flatten(),

    Dense(64,activation='relu'),
    Dense(64,activation='relu'),
    Dense(2,activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)



C:\Users\asus\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [6]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history=model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[early_stop]
)

Epoch 1/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 16s 139ms/step - accuracy: 0.8103 - loss: 0.4011 - val_accuracy: 0.9059 - val_loss: 0.2496
Epoch 2/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 11s 107ms/step - accuracy: 0.9243 - loss: 0.2194 - val_accuracy: 0.9291 - val_loss: 0.1877
Epoch 3/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 10s 98ms/step - accuracy: 0.9349 - loss: 0.1685 - val_accuracy: 0.9462 - val_loss: 0.1538
Epoch 4/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 11s 108ms/step - accuracy: 0.9615 - loss: 0.1189 - val_accuracy: 0.9499 - val_loss: 0.1783
Epoch 5/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 11s 111ms/step - accuracy: 0.9685 - loss: 0.0828 - val_accuracy: 0.9377 - val_loss: 0.2375
Epoch 6/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 12s 120ms/step - accuracy: 0.9710 - loss: 0.0799 - val_accuracy: 0.9584 - val_loss: 0.1711
Epoch 7/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 13s 130ms/step - accuracy: 0.9801 - loss: 0.0572 - val_accuracy: 0.9572 - val_loss: 0.1501
Epoch 8/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 13s 127ms/step - accuracy: 0.9890 - loss: 0.

In [11]:
img = tf.keras.utils.load_img(
    r"C:\Users\asus\Downloads\mask.webp",
    target_size=(128,128,)
    
)
class_names=['with mask','Without mask']
img_array = tf.keras.utils.img_to_array(img)
img_array = img_array 

img_array = np.expand_dims(img_array,axis=0)

prediction = model.predict(img_array)

print(prediction)
print(class_names)
print(np.max(prediction))
if np.max(prediction) < 0.60:
    print("Not a related image")
else:
    print("Predicted Expression :", class_names[np.argmax(prediction)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
[[9.9999428e-01 5.7686134e-06]]
['with mask', 'Without mask']
0.9999943
Predicted Expression : with mask


In [10]:
model.save("mask_emotion_new.h5")

In [1]:
import cv2
import numpy as np
import tensorflow as tf
model = tf.keras.models.load_model("mask_emotion.h5")
class_names = [
    'with mask',
    'without mask'
]
face_detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
print(face_detector.empty())
cap = cv2.VideoCapture(0)
while True:

    ret, frame = cap.read()

    if not ret:
        break

    gray=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
    faces = face_detector.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    for (x, y, w, h) in faces:

        face = frame[y:y+h, x:x+w]

        face = cv2.resize(face, (128,128))

        face = face.astype("float32")

        face = np.expand_dims(face, axis=0)

        prediction = model.predict(face, verbose=0)
        
        print(prediction)

        emotion = class_names[np.argmax(prediction)]
        print(np.max(prediction))
        confidence = np.max(prediction) * 100

        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            (0,255,0),
            2
        )

        cv2.putText(
            frame,
            f"{emotion} ({confidence:.1f}%)",
            (x, y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0,255,0),
            2
        )

    cv2.imshow("Emotion Detection", frame)
    

    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()

cv2.destroyAllWindows()

C:\Users\asus\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


False
[[1.0000000e+00 3.0839125e-20]]
1.0
[[1.000000e+00 9.093638e-26]]
1.0
[[1.0000000e+00 2.7288163e-23]]
1.0
[[1.0000000e+00 2.6363262e-24]]
1.0
[[1.0000000e+00 4.2495715e-25]]
1.0
[[1.0000000e+00 2.8182832e-25]]
1.0
[[1.00000e+00 5.95256e-22]]
1.0
[[1.0000000e+00 2.9748753e-23]]
1.0
[[1.00000000e+00 1.01736365e-25]]
1.0
[[1.0000000e+00 3.4495474e-23]]
1.0
[[1.0000000e+00 7.8835905e-26]]
1.0
[[1.000000e+00 2.849047e-25]]
1.0
[[1.0000000e+00 2.9128044e-27]]
1.0
[[1.0000000e+00 2.6681506e-28]]
1.0
[[1.0000000e+00 2.7780618e-26]]
1.0
[[1.000000e+00 9.496976e-29]]
1.0
[[1.000000e+00 3.363867e-26]]
1.0
[[1.0000000e+00 4.2883758e-26]]
1.0
[[1.0000000e+00 7.8728264e-27]]
1.0
[[1.000000e+00 6.529122e-26]]
1.0
[[1.0000000e+00 5.8742485e-27]]
1.0
[[1.0000000e+00 1.2990902e-25]]
1.0
[[1.0000000e+00 1.6994332e-26]]
1.0
[[1.0000000e+00 3.1024035e-27]]
1.0
[[1.0000000e+00 6.2103364e-28]]
1.0
[[1.00000000e+00 1.37328425e-27]]
1.0
[[1.0000000e+00 2.4005624e-28]]
1.0
[[1.0000000e+00 3.1931402e-27]]


In [2]:
import cv2
import numpy as np
import tensorflow as tf

# Load model
model = tf.keras.models.load_model("mask_emotion.h5")

class_names = [
    "with mask",
    "without mask"
]

# Face detector
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# Webcam
cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Haar Cascade requires grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_detector.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5,
        minSize=(60, 60)
    )

    for (x, y, w, h) in faces:

        # Crop face from original color image
        face = frame[y:y+h, x:x+w]

        # Convert BGR -> RGB (important)
        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)

        # Resize
        face = cv2.resize(face, (128, 128))

        # Convert to float
        face = face.astype(np.float32)

        # Do NOT divide by 255 if your model already has
        # Rescaling(1./255) as the first layer.

        # Add batch dimension
        face = np.expand_dims(face, axis=0)

        # Prediction
        prediction = model.predict(face, verbose=0)[0]

        predicted_class = np.argmax(prediction)
        confidence = prediction[predicted_class] * 100

        label = class_names[predicted_class]

        

        # Draw rectangle
        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        # Draw label
        cv2.putText(
            frame,
            f"{label} ({confidence:.1f}%)",
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

    cv2.imshow("Mask Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()